# 03 — The address join (plan Part 4)

**The hardest engineering problem in the project, and the one that caps
everything downstream.** Neither Price Paid nor EPC shares a key.

- **Route A** — UBDC's published Price-Paid-to-UPRN lookup (96% match rate,
  Jan 1995-Jan 2022), joined to EPC's own `UPRN` column.
- **Route B** — our own postcode + house-number matcher, with fuzzy
  street-name fallback. Needed because Route A **stops at January 2022**,
  and 2022-onward sales are the most valuable rows for pricing today's
  market.
- **Route A grades Route B** on the 2011-2022 overlap, so we get a measured
  precision/recall instead of "I hope the matcher works" before trusting it
  on the recent sales where nothing exists to check against.

Requires `data/raw/ubdc/ppdid_uprn_usrn.csv` and the cleaned outputs of
notebooks 01 and 02.

**UBDC file schema, confirmed against the real download**: `uprn,
transactionid, parentuprn, usrn`. It's the whole of England & Wales
(~30m rows, 1.6 GB) with no geography column to pre-filter on, so we filter
it with **DuckDB against our own Sefton transaction IDs** rather than
loading it into pandas — same pattern as notebook 01. `transactionid` uses
the same `{GUID}` format as Price Paid's own `transaction_id` (verified: both
look like `{79621BF5-5CAF-427F-983A-66CFECBA6450}`), so this is a direct
join, no parsing needed.


In [1]:
import sys
sys.path.insert(0, "..")
from src.config import DATA_RAW, DATA_INTERIM
import pandas as pd
import numpy as np
import re
import duckdb
from rapidfuzz import fuzz, process

ppd = pd.read_parquet(DATA_INTERIM / "price_paid_sefton_clean.parquet")
epc = pd.read_parquet(DATA_INTERIM / "epc_sefton_clean.parquet")
print(f"PPD: {len(ppd):,} transactions | EPC: {len(epc):,} certificates")


PPD: 64,348 transactions | EPC: 116,070 certificates


## Normalise the Price Paid side

EPC was normalised in notebook 02 (`address_norm`, `house_number`,
`number_suffix`, `sub_unit`). Price Paid carries the equivalent information
split across PAON (primary addressable object — usually the house number or
name) and SAON (secondary — flat/unit).


In [2]:
def normalise_ppd_address(row) -> dict:
    paon = str(row.get("paon", "") or "")
    saon = str(row.get("saon", "") or "")

    sub_unit = None
    if saon and saon != "nan":
        m = re.search(r"(\d+)", saon.upper())
        sub_unit = f"FLAT{m.group(1)}" if m else saon.upper().strip()

    m_num = re.search(r"\b(\d+)([A-Z]?)\b", paon.upper())
    house_number = m_num.group(1) if m_num else None
    number_suffix = m_num.group(2) if m_num and m_num.group(2) else None

    return {"house_number": house_number, "number_suffix": number_suffix, "sub_unit": sub_unit}

ppd_norm = ppd.apply(normalise_ppd_address, axis=1, result_type="expand")
ppd = pd.concat([ppd, ppd_norm], axis=1)
ppd["street_norm"] = ppd["street"].fillna("").str.upper().str.strip()
print(f"House number extracted for {ppd['house_number'].notna().mean():.1%} of PPD rows")


House number extracted for 94.6% of PPD rows


## Route A — UPRN via the UBDC lookup

Filtered with DuckDB against our ~64k Sefton `transaction_id`s rather than
loading all ~30m England & Wales rows into pandas.


In [3]:
ubdc_path = DATA_RAW / "ubdc" / "ppdid_uprn_usrn.csv"

if ubdc_path.exists():
    con = duckdb.connect()
    con.register("ppd_ids", ppd[["transaction_id"]])
    ubdc_matched = con.execute(f'''
        SELECT ubdc.transactionid AS transaction_id, ubdc.uprn AS uprn_route_a
        FROM read_csv('{ubdc_path.as_posix()}',
                       header = true,
                       columns = {{'uprn': 'BIGINT', 'transactionid': 'VARCHAR',
                                   'parentuprn': 'VARCHAR', 'usrn': 'VARCHAR'}}) AS ubdc
        JOIN ppd_ids ON ubdc.transactionid = ppd_ids.transaction_id
    ''').df()
    print(f"UBDC: {len(ubdc_matched):,} of {len(ppd):,} Sefton transaction_ids found a UPRN")

    ppd_a = ppd.merge(ubdc_matched, on="transaction_id", how="left")
    route_a_rate = ppd_a["uprn_route_a"].notna().mean()
    print(f"Route A matched {route_a_rate:.1%} of Sefton transactions to a UPRN")
else:
    print(f"MISSING: {ubdc_path}")
    print("Register at data.ubdc.ac.uk and download the lookup (notebook 00, step 4).")
    print("Continuing with Route B only for now.")
    ppd_a = ppd.copy()
    ppd_a["uprn_route_a"] = pd.NA


UBDC: 45,358 of 64,348 Sefton transaction_ids found a UPRN
Route A matched 70.5% of Sefton transactions to a UPRN


## Route B — our own matcher

1. Exact match on `(postcode, house_number, sub_unit)`.
2. For anything unmatched, fuzzy street-name match within the same postcode
   and house number (`rapidfuzz`, threshold tunable — start at 85).

⚠️ **26.1% of UPRNs have more than one EPC certificate** (notebook 02), so
merging straight against the full EPC table fans out — one PPD row would
match several EPC rows sharing the same address key, silently inflating the
row count past the original transaction total. We only need the
address→UPRN mapping here (which *certificate* is current as of the sale
gets decided in notebook 04), so **dedupe EPC to one row per address key
before merging**.

⚠️ **Keep every PPD row through this step, including the 5.4% without a
parseable house number** — merging on a `NaN` key just produces no match
(pandas never equates `NaN == NaN`), so those rows correctly fall through
as unmatched rather than being silently dropped. Dropping them first would
have discarded any Route A matches they already had.


In [4]:
epc_key = epc.dropna(subset=["house_number"]).copy()
epc_key["sub_unit"] = epc_key["sub_unit"].fillna("")
epc_key = epc_key.drop_duplicates(subset=["POSTCODE", "house_number", "sub_unit"])

ppd_key = ppd_a.copy()
ppd_key["sub_unit"] = ppd_key["sub_unit"].fillna("")

exact = ppd_key.merge(
    epc_key[["UPRN", "POSTCODE", "house_number", "sub_unit", "address_norm"]],
    left_on=["postcode", "house_number", "sub_unit"],
    right_on=["POSTCODE", "house_number", "sub_unit"],
    how="left",
    suffixes=("", "_epc"),
)
assert len(exact) == len(ppd_key), f"Join fanned out: {len(exact):,} rows from {len(ppd_key):,} input transactions"
exact_rate = exact["UPRN"].notna().mean()
print(f"Route B exact (postcode, house_number, sub_unit): {exact_rate:.1%} matched")


Route B exact (postcode, house_number, sub_unit): 75.7% matched


In [5]:
def fuzzy_fallback(row, epc_by_postcode, threshold=85):
    if pd.notna(row.get("UPRN")):
        return row["UPRN"], row.get("address_norm")
    candidates = epc_by_postcode.get(row["postcode"], None)
    if candidates is None or not row.get("house_number"):
        return None, None
    same_number = candidates[candidates["house_number"] == row["house_number"]]
    if same_number.empty:
        return None, None
    match = process.extractOne(
        row["street_norm"], same_number["address_norm"].tolist(), scorer=fuzz.token_sort_ratio
    )
    if match and match[1] >= threshold:
        matched_row = same_number.iloc[match[2]]
        return matched_row["UPRN"], matched_row["address_norm"]
    return None, None

epc_by_postcode = {pc: grp for pc, grp in epc_key.groupby("POSTCODE")}

fuzzy_results = exact.apply(lambda r: fuzzy_fallback(r, epc_by_postcode), axis=1, result_type="expand")
exact["uprn_route_b"] = exact["UPRN"].combine_first(fuzzy_results[0])
exact["matched_address_b"] = exact["address_norm"].combine_first(fuzzy_results[1])

route_b_rate = exact["uprn_route_b"].notna().mean()
print(f"Route B total (exact + fuzzy fallback): {route_b_rate:.1%} matched")


Route B total (exact + fuzzy fallback): 76.2% matched


## Grade Route B against Route A

On the 2011-2022 overlap where Route A has a ground-truth UPRN, check how
often Route B's independently-derived match agrees. This is the project's
**first real measured result** — everything downstream depends on it.


In [6]:
overlap = exact[
    exact["date_of_transfer"].between("2011-01-01", "2022-01-31")
    & exact["uprn_route_a"].notna()
]

if len(overlap):
    both_matched = overlap["uprn_route_b"].notna()
    agree = (overlap.loc[both_matched, "uprn_route_b"] == overlap.loc[both_matched, "uprn_route_a"])
    precision = agree.mean() if both_matched.any() else float("nan")
    recall = both_matched.mean()
    print(f"Route B precision (agrees with Route A when both match): {precision:.1%}")
    print(f"Route B recall (found *a* match where Route A did): {recall:.1%}")
else:
    print("No Route-A-matched rows in the overlap window yet - run Route A first.")


Route B precision (agrees with Route A when both match): 96.1%
Route B recall (found *a* match where Route A did): 76.7%


## Match rate by year and property type

The headline number for this notebook.


In [7]:
exact["matched_any"] = exact["uprn_route_a"].notna() | exact["uprn_route_b"].notna()
exact["year"] = exact["date_of_transfer"].dt.year

by_year = exact.groupby("year")["matched_any"].mean()
by_type = exact.groupby("property_type")["matched_any"].mean()
print("By year:")
print(by_year)
print("\nBy property type:")
print(by_type)
print(f"\nOverall: {exact['matched_any'].mean():.1%}")


By year:
year
2008    0.979806
2009    0.988101
2010    0.985269
2011    0.986604
2012    0.992531
2013    0.992955
2014    0.993381
2015    0.989775
2016    0.989864
2017    0.993908
2018    0.992031
2019    0.990262
2020    0.975706
2021    0.949613
2022    0.849350
2023    0.856246
2024    0.864787
2025    0.882149
2026    0.851817
Name: matched_any, dtype: float64

By property type:
property_type
D    0.986679
F    0.754142
S    0.993495
T    0.985885
Name: matched_any, dtype: float64

Overall: 95.3%


## Save the joined dataset


In [8]:
final_uprn = exact["uprn_route_a"].combine_first(exact["uprn_route_b"])
exact["uprn_final"] = final_uprn
joined = exact[exact["uprn_final"].notna()].copy()

out = DATA_INTERIM / "ppd_epc_joined_sefton.parquet"
joined.to_parquet(out, index=False)
print(f"Saved {len(joined):,} joined rows ({len(joined) / len(exact):.1%} of Sefton transactions) to {out}")


Saved 61,344 joined rows (95.3% of Sefton transactions) to C:\Users\jrbah\Documents (local)\Projects\house_price_prediction\data\interim\ppd_epc_joined_sefton.parquet
